# 06 · Who gains most from a top slot (Q2), and does the promotion badge work (Q3)?

**Q2:** fit a causal forest on the observational arm and compare group effects with the same groups in
the randomly ranked arm. **Q3:** estimate the promotion badge's effect inside the randomly ranked arm,
where position is random and so cannot confound it. Q3 has no experimental truth, so its verdict is at
most *suggestive, not experimentally verified*.

The forest here uses 60,000 rows for speed; `python -m causallab q2` uses the config value.

## Setup

Imports and configuration. Every threshold and setting comes from `config.yaml`. Set `SYNTHETIC = True` to run on the synthetic file built by `python -m causallab prepare --synthetic`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from causallab.config import load_config, load_spec, results_dir

cfg = load_config()
SYNTHETIC = False   # True = run on the synthetic Expedia-like file (no Kaggle data needed)
pd.set_option("display.width", 140)

## Q2: the causal forest

**What:** EconML `CausalForestDML` estimates an effect for every row as a function of the effect
modifiers (within-search price tercile, star band, trip type), adjusting for the other covariates.
Group effects are averages within each group.

**Why:** a 'which listings gain most from a boost' policy needs effects by segment, not one average.

**Expect:** a few minutes. With 60,000 rows the forest is noisy: its intervals are wide, and the
order of groups can differ from the full 1.2-million-row run shown in the next cell.

In [2]:
from causallab import data, features
from causallab.heterogeneity import run_q2
spec = load_spec(cfg)
df = features.q1_rows(data.load_arm("observational", cfg, SYNTHETIC))
q2 = run_q2(df, spec["adjustment_set"], cfg, SYNTHETIC, rows=60000)
rows = []
for gr in q2["groups"]:
    ex = gr["experimental"] or {}
    rows.append({"group": f"{gr['modifier']}={gr['level']}", "forest_pp": gr["estimate"] * 100,
                 "forest_low": gr["ci_low"] * 100, "forest_high": gr["ci_high"] * 100,
                 "experimental_pp": ex.get("estimate", np.nan) * 100 if ex else np.nan})
print("Small forest (60,000 rows) for speed: read the intervals, not just the points.")
pd.DataFrame(rows).round(2)

Small forest (60,000 rows) for speed: read the intervals, not just the points.


,group,forest_pp,forest_low,forest_high,experimental_pp
0,price_tercile=1,7.30,3.44,11.15,6.18
1,price_tercile=2,6.75,3.10,10.39,4.98
2,price_tercile=3,4.73,1.96,7.50,4.45
3,star_band=1-2,6.69,2.39,10.99,5.29
4,star_band=3,7.03,3.95,10.11,5.00
5,star_band=4,5.62,2.46,8.78,5.66
6,star_band=5,5.04,0.90,9.19,4.29
7,star_band=unrated,6.27,2.35,10.19,7.18
8,trip_type=family,7.69,4.15,11.23,5.44
9,trip_type=other,5.62,2.42,8.82,5.01


## The full run (1.2 million rows)

**What:** the saved result of `python -m causallab q2`, which the README reports.

**Expect:** on the real sample, the forest above the experiment in every group, by about 1.2 pp on average.

In [3]:
p = results_dir(cfg) / ("q2_group_effects_synthetic.json" if SYNTHETIC else "q2_group_effects.json")
full = json.loads(p.read_text(encoding="utf-8")) if p.exists() else {"groups": []}
if not full["groups"]:
    print("Not run yet: `python -m causallab q2` writes it.")
pd.DataFrame([{"group": f"{g['modifier']}={g['level']}", "forest_pp": g["estimate"] * 100,
               "forest_low": g["ci_low"] * 100, "forest_high": g["ci_high"] * 100,
               "experimental_pp": (g["experimental"] or {}).get("estimate", np.nan) * 100}
              for g in full["groups"]]).round(2)

,group,forest_pp,forest_low,forest_high,experimental_pp
0,price_tercile=1,7.88,6.81,8.95,6.18
1,price_tercile=2,6.70,5.76,7.65,4.98
2,price_tercile=3,5.08,4.26,5.90,4.45
3,star_band=1-2,7.00,5.87,8.13,5.29
4,star_band=3,6.91,6.14,7.67,5.00
5,star_band=4,6.43,5.77,7.10,5.66
6,star_band=5,4.78,3.52,6.04,4.29
7,star_band=unrated,7.77,4.87,10.66,7.18
8,trip_type=family,6.93,5.95,7.90,5.44
9,trip_type=other,6.42,5.55,7.30,5.01


## Reading the Q2 comparison

The experimental column comes from the randomly ranked arm (via the benchmark module). Where the forest
tracks it, the ranking of segments is credible; where it does not, the same confounding that biased Q1
is likely at work within that segment. The forest intervals are not clustered by search.

## Q3: the promotion badge, inside the randomly ranked arm

**What:** the five estimators with `promotion_flag` as treatment, on all positions of the randomly
ranked arm, adjusting for hotel and search covariates, with position as a precision covariate.

**Why:** promotion is not randomised, so it is confounded by hotel type and the hotel's usual price
level (`prop_log_historical_price` stays in). The *displayed* price is different: a promotion lowers it,
so displayed price and the features built from it are mediators, part of how the badge works. They are
left out, as Claude also did in the agent's Q3 run. Claude's final clicks run left out more (the
page-relative quality, affinity and visitor-history columns: 30 variables against 39 here), so the two
numbers can differ slightly. Working inside the random arm removes position as a confounder.

**Expect:** about 1.4 pp on clicks on the real sample, close to the agent's Q3 run (1.35 pp).

In [4]:
from causallab import estimators as E, refuters as R
exp = data.load_arm("experimental", cfg, SYNTHETIC)
# The badge lowers the displayed price: displayed price and features built from it are mediators
MEDIATORS = ["log_price", "price_rank_pct", "price_rel_median", "comp_expedia_cheaper_n", "comp_expedia_dearer_n"]
cov3 = [c for c in spec["adjustment_set"] if c not in ("promotion_flag", *MEDIATORS)] + ["position"]
q3 = E.estimate_all(exp, "promotion_flag", "click_bool", cov3, "srch_id", cfg)
pd.DataFrame({k: {"pp": v.estimate * 100, "low": v.ci_low * 100, "high": v.ci_high * 100}
              for k, v in q3["estimates"].items()}).T.round(2)

,pp,low,high
naive_untrimmed,1.67,1.58,1.76
naive,1.67,1.57,1.76
regression,1.51,1.42,1.60
ipw,1.40,1.29,1.51
aipw,1.37,1.27,1.47
dml,1.36,1.26,1.46


## What adjusting for a mediator does

**What:** the same doubly robust estimate, with displayed price and the features built from it put back in.

**Why:** adjusting for a mediator blocks part of the effect you are trying to measure: here, the part of
the badge's effect that works through the lower price.

**Expect:** a smaller number, roughly half (about 0.7 pp on the real sample). Treat this as an
illustration of what conditioning on a mediator does, not as a measured share of the effect that runs
through price: displayed price also partly reflects the hotel's own price level, which is a confounder.

In [5]:
q3_med = E.estimate_all(exp, "promotion_flag", "click_bool", cov3 + MEDIATORS, "srch_id", cfg, which=("aipw",))
a, b = q3["estimates"]["aipw"].estimate, q3_med["estimates"]["aipw"].estimate
print(f"displayed price left out: {a * 100:.2f} pp | displayed price adjusted for: {b * 100:.2f} pp")

displayed price left out: 1.37 pp | displayed price adjusted for: 0.73 pp


## Q3 sensitivity, reported prominently

**What:** the same hidden-confounder bounds as in notebook 04.

**Why:** with no randomised truth, sensitivity is the main evidence about how fragile the estimate is.

**Expect:** the verdict wording: *suggestive, not experimentally verified*.

In [6]:
k = exp[q3["keep"]].reset_index(drop=True)
s3 = R.sensitivity(k.click_bool.to_numpy(float), k.promotion_flag.to_numpy(), k[cov3], q3["estimates"]["aipw"], cfg, groups=k.srch_id.to_numpy())
print("passed:", s3["passed"], "| strongest:", s3["strongest_benchmark_covariate"], "| RV:", round(s3["robustness_value_q1"], 3))
print("Verdict: suggestive, not experimentally verified.")

passed: True | strongest: n_results | RV: 0.028
Verdict: suggestive, not experimentally verified.
